# Available accelerometer data

Review the catalog in chronological Colombia time (`America/Bogota`, UTC-05:00).
N/E/Z together count as one recording set. Gaps divide sets into continuous intervals;
daily and individual exports can overlap, so neither count proves independent field measurements.
Physical sites remain unknown. This notebook reads the catalog without changing raw files.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

working_dir = Path.cwd().resolve()
ROOT = next((folder for folder in (working_dir, *working_dir.parents)
             if (folder / 'data/accelerometer/catalog/intervals.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run from this repository or one of its subfolders.')

catalog_dir = ROOT / 'data/accelerometer/catalog'
intervals = pd.read_csv(catalog_dir / 'intervals.csv',
                        dtype={'location_code': str, 'network': str, 'station': str})
required = {'session_id', 'date_utc', 'source_kind', 'input_files', 'start_bogota',
            'end_bogota', 'duration_s', 'physical_site', 'loader_status',
            'time_selected_windows', 'profile_status'}
missing = required - set(intervals.columns)
if missing:
    raise ValueError(f'Catalog is missing columns: {sorted(missing)}')
if intervals.empty or intervals['session_id'].duplicated().any():
    raise ValueError('Expected a nonempty catalog with unique session IDs.')
for column in ('start_bogota', 'end_bogota'):
    intervals[column] = pd.to_datetime(intervals[column], format='mixed', utc=True,
                                       errors='raise').dt.tz_convert('America/Bogota')
if intervals[['start_bogota', 'end_bogota', 'input_files']].isna().any().any():
    raise ValueError('Catalog contains missing times or input paths.')
if (intervals['end_bogota'] <= intervals['start_bogota']).any():
    raise ValueError('Catalog contains invalid interval boundaries.')

intervals = intervals.sort_values(['start_bogota', 'end_bogota', 'session_id'],
                                 kind='stable').reset_index(drop=True)
input_paths = {path for files in intervals['input_files'] for path in files.split(';')}
missing_files = sorted(path for path in input_paths if not (ROOT / path).is_file())
if missing_files:
    raise FileNotFoundError(f'Catalog input files not found: {missing_files}')
intervals['duration_min'] = intervals['duration_s'] / 60
assert intervals['start_bogota'].is_monotonic_increasing


In [2]:
recording_sets = (
    intervals.groupby(['source_kind', 'input_files'], sort=False, dropna=False)
    .agg(start_bogota=('start_bogota', 'min'),
         end_bogota=('end_bogota', 'max'),
         continuous_intervals=('session_id', 'size'),
         physical_site=('physical_site', 'first'))
    .reset_index()
    .sort_values(['start_bogota', 'end_bogota', 'input_files'], kind='stable')
    .reset_index(drop=True)
)
recording_sets['date_bogota'] = recording_sets['start_bogota'].dt.date
recording_sets['recording_set'] = recording_sets['input_files'].map(
    lambda files: Path(files.split(';')[0]).parent.name + ' daily N/E/Z'
    if len(files.split(';')) == 3 else Path(files).stem)

print(f'{len(input_paths)} unique signal files; {len(recording_sets)} N/E/Z recording sets; '
      f'{len(intervals)} continuous intervals.')
print('Overlapping exports are counted separately; physical measurement sites are unknown.')
print('Recording-set start/end are coverage bounds, not uninterrupted durations.')
sets_by_date = pd.crosstab(recording_sets['date_bogota'], recording_sets['source_kind'])
sets_by_date['total_recording_sets'] = sets_by_date.sum(axis=1)
sets_by_date['continuous_intervals'] = intervals.groupby(
    intervals['start_bogota'].dt.date).size()
display(sets_by_date)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 70):
    display(recording_sets[['recording_set', 'source_kind', 'start_bogota',
                            'end_bogota', 'continuous_intervals', 'physical_site']])


32 unique signal files; 24 N/E/Z recording sets; 65 continuous intervals.
Overlapping exports are counted separately; physical measurement sites are unknown.
Recording-set start/end are coverage bounds, not uninterrupted durations.


source_kind,daily_triplet,individual_export,total_recording_sets,continuous_intervals
date_bogota,,,,
2023-05-18,1,0,1,3
2023-05-19,1,4,5,12
2023-05-20,1,11,12,36
2023-05-22,1,5,6,14


,recording_set,source_kind,start_bogota,end_bogota,continuous_intervals,physical_site
0,2023-05-18 daily N/E/Z,daily_triplet,2023-05-18 06:01:39.860000-05:00,2023-05-18 06:09:03.880000-05:00,3,unknown
1,2023-05-19 daily N/E/Z,daily_triplet,2023-05-19 16:33:20.570000-05:00,2023-05-19 18:55:51.355000-05:00,7,unknown
2,CBUCF_titanSMA_1614_20230519_213500,individual_export,2023-05-19 16:35:00-05:00,2023-05-19 16:59:42.835000-05:00,2,unknown
3,CBUCF_titanSMA_1614_20230519_221100,individual_export,2023-05-19 17:11:00-05:00,2023-05-19 17:34:09.450000-05:00,1,unknown
4,CBUCF_titanSMA_1614_20230519_224800,individual_export,2023-05-19 17:48:00-05:00,2023-05-19 18:13:40.600000-05:00,1,unknown
5,CBUCF_titanSMA_1614_20230519_232900,individual_export,2023-05-19 18:29:00-05:00,2023-05-19 18:55:36.240000-05:00,1,unknown
6,2023-05-20 daily N/E/Z,daily_triplet,2023-05-20 05:36:39.235000-05:00,2023-05-20 16:54:44.605000-05:00,14,unknown
7,CBUCF_titanSMA_1614_20230520_103800,individual_export,2023-05-20 05:38:00-05:00,2023-05-20 06:01:13.980000-05:00,2,unknown
8,CBUCF_titanSMA_1614_20230520_110900,individual_export,2023-05-20 06:09:00-05:00,2023-05-20 06:32:17.450000-05:00,1,unknown
9,CBUCF_titanSMA_1614_20230520_113700,individual_export,2023-05-20 06:37:00-05:00,2023-05-20 06:58:04.450000-05:00,1,unknown


## Continuous intervals sorted by `start_bogota`

The following table keeps only individual-export intervals strictly longer than 5 minutes, earliest first.
Change `MIN_DURATION_MIN` to adjust this threshold. Raw data and the full summary remain unchanged.
Set `DATE_BOGOTA` to a date
such as `'2023-05-20'`, or `SOURCE_KIND` to `'individual_export'` or `'daily_triplet'`
to narrow the view. Use the original `session_id` to select notebook inputs.
`time_selected_windows` and `profile_status` reflect the profile stored when the
catalog was generated, not a new HVSR calculation. Regenerate the catalog after
changing processing parameters.

In [3]:
DATE_BOGOTA = None
SOURCE_KIND = 'individual_export'
MIN_DURATION_MIN = 5.0

if not 0 <= MIN_DURATION_MIN < float('inf'):
    raise ValueError('MIN_DURATION_MIN must be finite and nonnegative.')
review = intervals.loc[intervals['duration_s'] > MIN_DURATION_MIN * 60].copy()
if DATE_BOGOTA is not None:
    requested_date = pd.Timestamp(DATE_BOGOTA).date()
    review = review.loc[review['start_bogota'].dt.date == requested_date]
if SOURCE_KIND is not None:
    if SOURCE_KIND not in set(intervals['source_kind']):
        raise ValueError(f'Unknown source kind: {SOURCE_KIND}')
    review = review.loc[review['source_kind'] == SOURCE_KIND]
assert (review['duration_s'] > MIN_DURATION_MIN * 60).all()
assert review['start_bogota'].is_monotonic_increasing
assert SOURCE_KIND is None or review['source_kind'].eq(SOURCE_KIND).all()
print(f'Showing {len(review)} of {len(intervals)} intervals longer than '
      f'{MIN_DURATION_MIN:g} minutes, source={SOURCE_KIND}, sorted by start_bogota.')
if review.empty:
    print('No intervals match the selected filters.')
columns = ['session_id', 'source_kind', 'start_bogota', 'end_bogota', 'duration_min',
           'physical_site', 'loader_status', 'time_selected_windows', 'profile_status']
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.max_colwidth', 65):
    display(review[columns].round({'duration_min': 3}))


Showing 20 of 65 intervals longer than 5 minutes, source=individual_export, sorted by start_bogota.


,session_id,source_kind,start_bogota,end_bogota,duration_min,physical_site,loader_status,time_selected_windows,profile_status
6,CBUCF_titanSMA_1614_20230519_213500_02,individual_export,2023-05-19 16:35:24.665000-05:00,2023-05-19 16:59:42.835000-05:00,24.303,unknown,verified,0,fewer_than_two_time_selected_windows
9,CBUCF_titanSMA_1614_20230519_221100_01,individual_export,2023-05-19 17:11:00-05:00,2023-05-19 17:34:09.450000-05:00,23.158,unknown,verified,0,fewer_than_two_time_selected_windows
11,CBUCF_titanSMA_1614_20230519_224800_01,individual_export,2023-05-19 17:48:00-05:00,2023-05-19 18:13:40.600000-05:00,25.677,unknown,verified,0,fewer_than_two_time_selected_windows
14,CBUCF_titanSMA_1614_20230519_232900_01,individual_export,2023-05-19 18:29:00-05:00,2023-05-19 18:55:36.240000-05:00,26.604,unknown,verified,0,fewer_than_two_time_selected_windows
18,CBUCF_titanSMA_1614_20230520_103800_02,individual_export,2023-05-20 05:38:44.295000-05:00,2023-05-20 06:01:13.980000-05:00,22.495,unknown,verified,0,fewer_than_two_time_selected_windows
20,CBUCF_titanSMA_1614_20230520_110900_01,individual_export,2023-05-20 06:09:00-05:00,2023-05-20 06:32:17.450000-05:00,23.291,unknown,verified,0,fewer_than_two_time_selected_windows
22,CBUCF_titanSMA_1614_20230520_113700_01,individual_export,2023-05-20 06:37:00-05:00,2023-05-20 06:58:04.450000-05:00,21.074,unknown,verified,0,fewer_than_two_time_selected_windows
24,CBUCF_titanSMA_1614_20230520_120239_02,individual_export,2023-05-20 07:03:04.625000-05:00,2023-05-20 07:26:18.815000-05:00,23.236,unknown,verified,0,fewer_than_two_time_selected_windows
26,CBUCF_titanSMA_1614_20230520_120500_01,individual_export,2023-05-20 07:05:00-05:00,2023-05-20 07:26:18.815000-05:00,21.314,unknown,verified,0,fewer_than_two_time_selected_windows
31,CBUCF_titanSMA_1614_20230520_123500_02,individual_export,2023-05-20 07:37:04.045000-05:00,2023-05-20 07:59:39.465000-05:00,22.590,unknown,verified,0,fewer_than_two_time_selected_windows
